# Pixels, Channels, Color, and Masks

> **Beginner · Image fundamentals**


## Why this matters

Most practical vision begins by selecting the right pixels. Regions of interest, channels, color spaces, and masks are the language for doing that precisely.

**Where it appears:** Color-based object selection, green-screen effects, inspections, UI overlays, and preprocessing for segmentation.


## Learning Objectives

- Read and write individual pixels and rectangular regions correctly and efficiently
- Split, merge, and manipulate individual color channels
- Use masks to select non-rectangular regions of interest
- Convert between BGR, HSV, LAB, and grayscale, and know when each is appropriate
- Perform robust color-based object selection using HSV thresholds
- Understand why HSV is preferred over BGR for color-based segmentation


## Prerequisites

02 NumPy for Images and 03 OpenCV Setup and Your First Pipeline

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

array slicing, `cv2.split`, `cv2.merge`, `cv2.cvtColor`, `cv2.inRange`, masks

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Pixels, ROI, and Channels

Direct pixel access (`image[y, x]`) is convenient but slow if used in a
Python loop over every pixel -- OpenCV/NumPy provide vectorized
alternatives for anything beyond a handful of pixels. Channel
splitting/merging and boolean masks are the building blocks for
color-based selection used throughout the segmentation and color-space
notebooks later in the series.


### Color Spaces and Color Conversions

BGR/RGB mix color and brightness together in every channel, which makes
color-based thresholding fragile under lighting changes. **HSV** (Hue,
Saturation, Value) separates *what color* (Hue) from *how vivid* (Saturation)
and *how bright* (Value), so thresholding on Hue alone is far more robust
to lighting. **LAB** separates perceptual lightness from color and is
useful for lighting-invariant comparisons (e.g. `L` channel for contrast
enhancement without shifting color, as used in CLAHE later).


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Pixels, ROI, and Channels


### 1. Single-pixel and small-region access

Direct indexing is fine for a few pixels (e.g. sampling colors) but should never be used inside a loop over a whole image -- that belongs to NumPy vectorized ops.


In [ ]:
import numpy as np
import cv2
from cv_utils import load_real_image, get_real_data


def sample_pixels(image: np.ndarray, points: list[tuple[int, int]]) -> dict:
    """Read BGR values at a handful of (x, y) points. Fine for sampling, NOT for full-image loops."""
    return {(x, y): tuple(int(v) for v in image[y, x]) for x, y in points}


scene = load_real_image("images/standard", "messi5.jpg")
samples = sample_pixels(scene, [(50, 50), (330, 100), (0, 0)])
for point, bgr in samples.items():
    print(f"pixel {point} -> BGR {bgr}")

### 2. Splitting and merging channels

`cv2.split`/`cv2.merge` separate and recombine channels. This is used to inspect individual color contributions or build false-color visualizations.


In [ ]:
def channel_breakdown(image: np.ndarray) -> dict[str, np.ndarray]:
    """Split a BGR image and return each channel as a single-channel image, labeled."""
    b, g, r = cv2.split(image)
    return {"Blue channel": b, "Green channel": g, "Red channel": r}


from cv_utils import load_real_image, get_real_data, show_grid

channels = channel_breakdown(scene)
show_grid(list(channels.items()))

# Merge back and confirm it's identical to the original
merged = cv2.merge(
    [channels["Blue channel"], channels["Green channel"], channels["Red channel"]]
)
print("Merge round-trip identical:", np.array_equal(scene, merged))

### 3. Mask-based (non-rectangular) ROI selection

A boolean/uint8 mask selects arbitrary-shaped regions -- not just rectangles. `cv2.bitwise_and` applies a mask; this pattern is the foundation of the later segmentation and background-subtraction notebooks.


In [ ]:
def circular_mask(
    shape: tuple[int, int], center: tuple[int, int], radius: int
) -> np.ndarray:
    """Build a single-channel 0/255 mask for a filled circle."""
    mask = np.zeros(shape, dtype=np.uint8)
    cv2.circle(mask, center, radius, 255, -1)
    return mask


mask = circular_mask(scene.shape[:2], (336, 286), 40)
masked_region = cv2.bitwise_and(scene, scene, mask=mask)

show_grid([("mask", mask), ("scene AND mask", masked_region)])

## Part 2: Color Spaces and Color Conversions


### 1. Converting between color spaces

`cv2.cvtColor` handles all standard conversions via flag constants. Build a small helper that converts to several spaces at once for comparison.


In [ ]:
import cv2
from cv_utils import load_real_image, get_real_data, show_grid


def to_all_spaces(image) -> dict:
    return {
        "BGR (original)": image,
        "Grayscale": cv2.cvtColor(image, cv2.COLOR_BGR2GRAY),
        "HSV": cv2.cvtColor(image, cv2.COLOR_BGR2HSV),
        "LAB": cv2.cvtColor(image, cv2.COLOR_BGR2LAB),
    }


scene = load_real_image("images/objects", "smarties.png")
spaces = to_all_spaces(scene)
# Note: HSV/LAB arrays shown here are NOT visually meaningful as RGB -- see next cell for a fair display
print({k: (v.shape, v.dtype) for k, v in spaces.items()})


### 2. HSV-based color selection

Threshold the Hue channel (with generous Saturation/Value bounds) to select all 'green-ish' pixels regardless of shading -- far more robust than thresholding BGR directly.


In [ ]:
import numpy as np


def select_color_hsv(
    image,
    hue_center: int,
    hue_tolerance: int = 15,
    sat_min: int = 60,
    val_min: int = 60,
) -> np.ndarray:
    """Return a binary mask of pixels close to `hue_center` (OpenCV hue range 0-179)."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lower = np.array([max(hue_center - hue_tolerance, 0), sat_min, val_min])
    upper = np.array([min(hue_center + hue_tolerance, 179), 255, 255])
    return cv2.inRange(hsv, lower, upper)


green_hue = int(cv2.cvtColor(np.uint8([[[90, 200, 90]]]), cv2.COLOR_BGR2HSV)[0, 0, 0])
green_mask = select_color_hsv(scene, hue_center=green_hue)
isolated_green = cv2.bitwise_and(scene, scene, mask=green_mask)

show_grid(
    [("original", scene), ("green mask", green_mask), ("isolated", isolated_green)]
)

### 3. Why HSV beats BGR for this task

Demonstrate robustness directly: darken the scene (simulating a lighting change) and confirm the HSV-based mask still finds most of the green region, while a naive BGR threshold would need re-tuning.


In [ ]:
def darken(image, factor: float = 0.5):
    return (image.astype(np.float32) * factor).astype(np.uint8)


dark_scene = darken(scene, 0.5)
dark_mask = select_color_hsv(dark_scene, hue_center=green_hue)

overlap = cv2.bitwise_and(green_mask, dark_mask)
recall = overlap.sum() / max(green_mask.sum(), 1)
print(
    f"Mask overlap after darkening the scene 50%: {recall:.1%} of original green pixels still detected"
)

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Pixels, ROI, and Channels: Multi-Spectral Channel Alignment (Image Registration)

In multi-camera systems or satellite imaging, different spectral bands (like Red, Green, Blue) can be slightly misaligned due to chromatic aberration or sensor shifts. Here we simulate a shift in the Red channel and align it back using Mean Squared Error (MSE) template matching.


In [ ]:
# Create a base grayscale shapes image
base = cv2.cvtColor(
    load_real_image("images/standard", "messi5.jpg"), cv2.COLOR_BGR2GRAY
)

# Simulate chromatic alignment shift: shift red channel by (dy, dx) = (3, -2) pixels
shifted_red = np.zeros_like(base)
shifted_red[3:, :-2] = base[:-3, 2:]

# Target search grid to realign shifted_red to base
best_dx, best_dy = 0, 0
min_mse = float("inf")

# Search translation offsets in range [-5, 5]
for dy in range(-5, 6):
    for dx in range(-5, 6):
        # Apply transformation to shift red channel
        temp = np.zeros_like(shifted_red)

        # Perform boundary-safe slicing
        ys, yd = max(0, -dy), max(0, dy)
        xs, xd = max(0, -dx), max(0, dx)
        h, w = base.shape
        lh, lw = h - max(abs(dy), 0), w - max(abs(dx), 0)

        if lh <= 0 or lw <= 0:
            continue

        temp[yd : yd + lh, xd : xd + lw] = shifted_red[ys : ys + lh, xs : xs + lw]

        # Calculate Mean Squared Error
        mse = np.mean((base - temp) ** 2)
        if mse < min_mse:
            min_mse = mse
            best_dy, best_dx = dy, dx

print(f"Simulated displacement offset: dy=3, dx=-2")
print(f"Estimated correction offset: dy={-best_dy}, dx={-best_dx}")

### Mini Project — Color Spaces and Color Conversions: Skin Color Segmentation in YCrCb Space

The YCrCb color space is widely used for skin segmentation because skin tones cluster closely in the Chrominance-Red (Cr) and Chrominance-Blue (Cb) channels, regardless of luminance (Y). This makes skin segmentation highly robust against lighting changes.


In [ ]:
# Load a real face image
face = load_real_image("images/faces", "face.jpg")

# Convert from BGR to YCrCb
ycrcb = cv2.cvtColor(face, cv2.COLOR_BGR2YCrCb)

# Defined boundaries for skin color in YCrCb
# Y range: 0-255 (we ignore to remain brightness-invariant)
# Cr range: 133 to 173, Cb range: 77 to 127
lower_skin = np.array([0, 133, 77], dtype=np.uint8)
upper_skin = np.array([255, 173, 127], dtype=np.uint8)

# Binarize to locate skin regions
skin_mask = cv2.inRange(ycrcb, lower_skin, upper_skin)

# Segment out skin
skin_only = cv2.bitwise_and(face, face, mask=skin_mask)

print("Face image shape:", face.shape)
print("Skin mask shape:", skin_mask.shape)
show_grid(
    [
        ("Original Face", face),
        ("Skin Mask", skin_mask),
        ("Skin Segments Only", skin_only),
    ]
)


## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Pixels, ROI, and Channels
1. Write `average_color_in_mask(image, mask)` returning the mean BGR color inside the masked region only.
2. Build a mask that is the *union* of two circles using `cv2.bitwise_or`.
3. Write a function that swaps the Red and Blue channels of an image using split/merge.

Use the empty cell below to work through them.


#### Solutions — Pixels, ROI, and Channels

In [ ]:
# Solution 1: average_color_in_mask
def average_color_in_mask(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Compute average BGR color of pixels within a boolean mask."""
    # Extract only pixel values where mask is true (shape: N x 3)
    matching_pixels = image[mask > 0]
    if len(matching_pixels) == 0:
        return np.array([0.0, 0.0, 0.0])
    return np.mean(matching_pixels, axis=0)


# Solution 2: Union of two circles using bitwise OR
def union_circles_mask(height: int, width: int) -> np.ndarray:
    """Generate a binary mask containing union of two circular elements."""
    mask1 = np.zeros((height, width), dtype=np.uint8)
    mask2 = np.zeros((height, width), dtype=np.uint8)

    cv2.circle(mask1, (width // 3, height // 2), 50, 255, -1)
    cv2.circle(mask2, (2 * width // 3, height // 2), 50, 255, -1)

    union_mask = cv2.bitwise_or(mask1, mask2)
    return union_mask


# Solution 3: Swapping Red and Blue channels using split/merge
def swap_red_blue(image: np.ndarray) -> np.ndarray:
    """Swap Red and Blue channels in place using split/merge."""
    b, g, r = cv2.split(image)
    return cv2.merge([r, g, b])


# Run validation checks
img = load_real_image("images/standard", "messi5.jpg")
mask = np.zeros(img.shape[:2], dtype=np.uint8)
cv2.rectangle(mask, (30, 30), (180, 160), 255, -1)
print("Average BGR color of red rectangle:", average_color_in_mask(img, mask))
union = union_circles_mask(200, 300)
print("Union mask shape:", union.shape)
swapped = swap_red_blue(img)
print("Swapped image BGR shape matches original:", swapped.shape == img.shape)

### Exercises — Color Spaces and Color Conversions
1. Write `select_color_hsv` variants for red, which requires wrapping around hue=0/179 (two ranges OR'd together).
2. Plot the LAB `L` channel alone and confirm it behaves like a lighting-only grayscale image.
3. Measure the same darkening-robustness experiment but thresholding directly on BGR values instead of HSV -- compare recall.

Use the empty cell below to work through them.


#### Solutions — Color Spaces and Color Conversions

In [ ]:
# Solution 1: select_color_hsv red wrapping range handling
def select_color_hsv_red(hsv_image: np.ndarray) -> np.ndarray:
    """Extract red pixels from HSV image, wrapping around hue limits (0 and 179)."""
    # Red wrap lower range
    lower1 = np.array([0, 70, 50])
    upper1 = np.array([10, 255, 255])
    mask1 = cv2.inRange(hsv_image, lower1, upper1)

    # Red wrap upper range
    lower2 = np.array([170, 70, 50])
    upper2 = np.array([179, 255, 255])
    mask2 = cv2.inRange(hsv_image, lower2, upper2)

    return cv2.bitwise_or(mask1, mask2)

In [ ]:

# Solution 2: Plot the LAB L channel
def plot_lab_l_channel(image: np.ndarray) -> None:
    """Plot the L (lightness) channel from the CIE L*a*b* color space."""
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2Lab)
    l_channel = lab[:, :, 0]
    show(l_channel, "CIE L*a*b* L-Channel (Lighting Only)", cmap="gray")

In [ ]:
# Solution 3: Darkening robustness experiment (BGR vs HSV thresholding)
# Explanation: Thresholding directly on BGR coordinates under shadow conditions is fragile.
# Shadows scale down the RGB values proportionally, causing the color coordinates to shift.
# Because HSV decouples chromatic components (Hue and Saturation) from luminance (Value),
# HSV thresholds can robustly match shadow-affected colored regions while BGR fails.

## Summary

You can select, visualize, and alter pixels safely, choosing a color representation that suits the task rather than guessing in BGR.

- **Best Practices:** Use vectorized ROIs and masks, document color-space assumptions, and inspect mask coverage before trusting a color rule.
- **Common Pitfalls:** Slow pixel loops, confusing channel order, overly broad HSV ranges, and modifying an image through an unintended view.